# DevRev Search Dataset

Loading and exploring the `devrev/search` dataset from Hugging Face.

**Dataset Structure:**
- `annotated_queries` — Queries paired with annotated (golden) article chunks
- `knowledge_base` — Article chunks from DevRev's customer-facing support documentation
- `test_queries` — Held-out queries used for evaluation

In [ ]:
%pip install -r requirements.txt


In [ ]:
from datasets import load_dataset
import pandas as pd
import voyageai
from dotenv import load_dotenv
load_dotenv()
import zvec
import numpy as np
from tqdm import tqdm
import time
import os
import json
import pickle
from rank_bm25 import BM25Okapi
from collections import defaultdict


## 1. Load Annotated Queries
Queries paired with annotated (golden) article chunks for training/validation.

In [4]:
# Load annotated queries
annotated_queries = load_dataset("devrev/search", "annotated_queries")
print(annotated_queries)

DatasetDict({
    train: Dataset({
        features: ['query_id', 'query', 'retrievals'],
        num_rows: 291
    })
})


In [5]:
# Convert to DataFrame and display
annotated_df = annotated_queries["train"].to_pandas()
annotated_df.head()

,query_id,query,retrievals
0,0ae94217-c6a0-4895-83a2-841a95f01637,create DevRev ticket from Microsoft Teams,"[{'id': 'ART-4216_KNOWLEDGE_NODE-26', 'text': ..."
1,d0b209b3-6cea-46d8-bfac-bd0e286ea21b,workflow builder auto close ticket after 48 ho...,"[{'id': 'ART-2012_KNOWLEDGE_NODE-24', 'text': ..."
2,40c1aa6f-cd21-46ab-8f6f-76fdc267b584,automated reminder to customer ticket will be ...,"[{'id': 'ART-3068_KNOWLEDGE_NODE-24', 'text': ..."
3,e47d883f-b712-4f98-bd06-14ade143e3c2,connect Bitbucket account to DevRev account,"[{'id': 'ART-2030_KNOWLEDGE_NODE-27', 'text': ..."
4,2e6f9413-15ac-4974-a380-7aa22fc98a61,use of workflows in DevRev,"[{'id': 'ART-1961_KNOWLEDGE_NODE-28', 'text': ..."


In [6]:
# Sample a single annotated query example
annotated_queries["train"][0]

{'query_id': '0ae94217-c6a0-4895-83a2-841a95f01637',
 'query': 'create DevRev ticket from Microsoft Teams',
 'retrievals': [{'id': 'ART-4216_KNOWLEDGE_NODE-26',
   'text': 'DevRev Object | Sync to DevRev |\\n| --- | --- | --- |\\n| Plan | Parts | \\xe2\\x9c\\x85 |\\n| User | Identity/DevUser | \\xe2\\x9c\\x85 |\\n| Channel | Chat | \\xe2\\x9c\\x85 |\\n| Attachments in Message/Thread/Task | Artifacts on Article | \\xe2\\x9c\\x85 |\\n| Message | Comment | \\xe2\\x9c\\x85 |\\n| Thread | Comment | \\xe2\\x9c\\x85 |\\n| Task | Issue/Ticket | \\xe2\\x9c\\x85 |\\n\\nImporting from Microsoft Teams\\n------------------------------\\n\\nFollow the steps below to import from Microsoft Teams:\\n\\n1. Go to',
   'title': 'Microsoft Teams AirSync | AirSync | Snap-ins | DevRev'},
  {'id': 'ART-4216_KNOWLEDGE_NODE-29',
   'text': 'with many\\nattachments. DevRev honors the Microsoft Graph API rate limits and back-off and resumes automatically.\\n\\nPost import options\\n-------------------\\n\\nAfter 

## 2. Load Test Queries
Held-out queries used for evaluation.

In [7]:
# Load test queries
test_queries = load_dataset("devrev/search", "test_queries")
print(test_queries)

DatasetDict({
    test: Dataset({
        features: ['query_id', 'query'],
        num_rows: 92
    })
})


In [8]:
# Convert to DataFrame and display
test_df = test_queries["test"].to_pandas()
test_df.head()

,query_id,query
0,a97f93d2-410a-431f-ae9a-1e23ed35d74c,end customer organization name not appearing i...
1,7dd7e2b4-9349-4535-8007-1d706e0fabff,Android SDK session generated with Unknown user
2,4bc92187-cdaa-4c20-b189-abd1672e5a71,email reply received on wrong ticket
3,4d9878e8-f746-4df5-8bf6-f9444989b385,manage access and privileges in DevRev
4,483151ec-aff4-4569-b3df-651f578b61d8,SSO setup SAML IDP metadata connection string ...


In [9]:
# Sample a single test query example
test_queries["test"][0]

{'query_id': 'a97f93d2-410a-431f-ae9a-1e23ed35d74c',
 'query': 'end customer organization name not appearing in ticket or conversation'}

## 3. Load Knowledge Base
Article chunks from DevRev's customer-facing support documentation.

In [10]:
# Load knowledge base
knowledge_base = load_dataset("devrev/search", "knowledge_base")
print(knowledge_base)

DatasetDict({
    corpus: Dataset({
        features: ['id', 'text', 'title'],
        num_rows: 65224
    })
})


In [11]:
# Convert to DataFrame and display
knowledge_df = knowledge_base["corpus"].to_pandas()
knowledge_df.head()

,id,text,title
0,ART-17711_KNOWLEDGE_NODE-0,b'We ran into a case where an AirSync was star...,Sync fails when original sync owners loses per...
1,ART-17711_KNOWLEDGE_NODE-1,access.\n\nOnce Person A was re-added with the...,Sync fails when original sync owners loses per...
2,ART-17650_KNOWLEDGE_NODE-0,"b""American cybersecurity leader unifies securi...",American cybersecurity leader unifies security...
3,ART-17650_KNOWLEDGE_NODE-1,DevRev\n======================================...,American cybersecurity leader unifies security...
4,ART-17650_KNOWLEDGE_NODE-2,solutions help organisations build and deploy ...,American cybersecurity leader unifies security...


In [12]:
# Sample a single knowledge base chunk
knowledge_base["corpus"][0]

{'id': 'ART-17711_KNOWLEDGE_NODE-0',
 'text': "b'We ran into a case where an AirSync was started by one person (Person A) and later failed. Another user (Person B) tried to click Retry, but it didn\\xe2\\x80\\x99t work. The logs showed 401 and 403 errors in communication between the snap-in and the snap-in manager.\\n\\nIt turned out that AirSync assigns the sync owner to whoever started it. Since Person A had been removed from the org or lost permissions, the retry failed \\xe2\\x80\\x94 the system still expected the original owner to have valid",
 'title': 'Sync fails when original sync owners loses permissions'}

## 4. Dataset Summary

In [13]:
print("=" * 60)
print("DevRev Search Dataset Summary")
print("=" * 60)
print(f"\nAnnotated Queries:")
print(annotated_queries)
print(f"\nTest Queries:")
print(test_queries)
print(f"\nKnowledge Base:")
print(knowledge_base)
print("\n" + "=" * 60)

DevRev Search Dataset Summary

Annotated Queries:
DatasetDict({
    train: Dataset({
        features: ['query_id', 'query', 'retrievals'],
        num_rows: 291
    })
})

Test Queries:
DatasetDict({
    test: Dataset({
        features: ['query_id', 'query'],
        num_rows: 92
    })
})

Knowledge Base:
DatasetDict({
    corpus: Dataset({
        features: ['id', 'text', 'title'],
        num_rows: 65224
    })
})



---
## 5. Index Knowledge Base with Zvec and BM25

Using Voyage for dense embeddings/reranking, Zvec for vector search, and BM25 for sparse retrieval.

In [ ]:
VOYAGE_API_KEY = os.environ.get("VOYAGE_API_KEY")
if not VOYAGE_API_KEY:
    print("Please set VOYAGE_API_KEY environment variable. Using dummy API key for offline execution.")
    VOYAGE_API_KEY = "DUMMY"
    
vo = voyageai.Client(api_key=VOYAGE_API_KEY)

MODEL_ID = "voyage-4-large"  
print(f"Using model: {MODEL_ID}")


Using model: voyage-4-large


In [ ]:
corpus = knowledge_base["corpus"]

documents = []
doc_ids = []
doc_titles = []
doc_texts = []

for item in tqdm(corpus, desc="Preparing documents"):
    doc_text = f"{item['title']}\n\n{item['text']}"
    documents.append(doc_text)
    doc_ids.append(item['id'])
    doc_titles.append(item['title'])
    doc_texts.append(item['text'])

print(f"\nTotal documents: {len(documents):,}")


Preparing documents: 100%|██████████| 65224/65224 [00:01<00:00, 51325.14it/s]


Total documents: 65,224


In [ ]:
print("Building BM25 index...")
tokenized_corpus = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 index built!")


Building BM25 index...
BM25 index built!


In [17]:
def get_all_embeddings(texts, batch_size=100):
    """Get embeddings for all texts using Voyage API."""
    all_embeddings = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Generating embeddings"):
        batch = texts[i:i + batch_size]
        try:
            response = vo.embed(batch, model=MODEL_ID, output_dimension=2048, input_type="document")
            all_embeddings.extend(response.embeddings)
        except Exception as e:
            print(f"Error: {e}")
            all_embeddings.extend([np.zeros(2048).tolist()] * len(batch))
        time.sleep(0.1)
    return np.array(all_embeddings)


In [ ]:
print("Generating or loading embeddings...")
embed_path = "voyage_embeddings.npy"
if os.path.exists(embed_path):
    embeddings = np.load(embed_path)
else:
    embeddings = get_all_embeddings(documents, batch_size=128)
    np.save(embed_path, embeddings)

print(f"Embeddings shape: {embeddings.shape}")


Generating or loading embeddings...
Embeddings shape: (65224, 2048)


In [ ]:
schema = zvec.CollectionSchema(
    name="devrev_kb",
    fields=[
        zvec.FieldSchema("title", zvec.DataType.STRING),
        zvec.FieldSchema("text", zvec.DataType.STRING),
        zvec.FieldSchema("doc_id", zvec.DataType.STRING)
    ],

    vectors=zvec.VectorSchema("embedding", zvec.DataType.VECTOR_FP32, 2048),
)

collection = zvec.create_and_open(path="./zvec_devrev", schema=schema)

print("Inserting into Zvec (this replaces faiss.Index.add)...")
batch_docs = []
for i, (doc_id, doc, title, text) in enumerate(zip(doc_ids, documents, doc_titles, doc_texts)):
    emb = embeddings[i]
    batch_docs.append(zvec.Doc(
        id=str(i),
        vectors={"embedding": emb.tolist()},
        fields={"doc_id": doc_id, "title": title, "text": text}
    ))
    if len(batch_docs) >= 1000:
        collection.insert(batch_docs)
        batch_docs = []
if batch_docs:
    collection.insert(batch_docs)

print("Successfully populated Zvec collection!")


Inserting into Zvec (this replaces faiss.Index.add)...
Successfully populated Zvec collection!


## 6. Hybrid Search Implementation

In [ ]:
def min_max_normalize(scores):
    min_val, max_val = np.min(scores), np.max(scores)
    if max_val == min_val:
        return [0 for _ in scores]
    return [(s - min_val) / (max_val - min_val) for s in scores]

def hybrid_search(query, k_recall=100, k_final=10, alpha=0.65):
    try:
        q_emb = vo.embed([query], model="voyage-4", output_dimension=2048, input_type="query").embeddings[0]
    except Exception:
        q_emb = np.zeros(2048).tolist()
        
    dense_results = collection.query(
        zvec.VectorQuery("embedding", vector=q_emb), topk=k_recall
    )
    dense_dict = {res.id: res.score for res in dense_results}
    
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_top_idx = np.argsort(bm25_scores)[::-1][:k_recall]
    bm25_dict = {idx: bm25_scores[idx] for idx in bm25_top_idx}
    
    all_candidates = set(dense_dict.keys()).union(set(bm25_dict.keys()))
    
    dense_vals = [dense_dict.get(c, 0) for c in all_candidates]
    bm25_vals = [bm25_dict.get(c, 0) for c in all_candidates]
    
    norm_dense = min_max_normalize(dense_vals)
    norm_bm25 = min_max_normalize(bm25_vals)
    
    candidate_items = list(all_candidates)
    fused_scores = { 
        candidate_items[i]: alpha * norm_dense[i] + (1 - alpha) * norm_bm25[i]
        for i in range(len(candidate_items))
    }
    
    fused_top_idx = sorted(candidate_items, key=lambda x: fused_scores[x], reverse=True)[:k_final]
    
    final_docs = [documents[int(idx)] for idx in fused_top_idx]
    
    try:
        rerank_results = vo.rerank(query, final_docs, model="rerank-2.5", top_k=k_final)
        final_results = []
        for r in rerank_results.results:
            orig_idx = fused_top_idx[r.index]
            final_results.append({
                "id": doc_ids[int(orig_idx)],
                "title": doc_titles[int(orig_idx)],
                "text": doc_texts[int(orig_idx)],
                "score": r.relevance_score
            })
        return final_results
    except Exception:
        final_results = []
        for orig_idx in fused_top_idx:
            final_results.append({
                "id": doc_ids[int(orig_idx)],
                "title": doc_titles[int(orig_idx)],
                "text": doc_texts[int(orig_idx)],
                "score": fused_scores[orig_idx]
            })
        return final_results


In [24]:
# Test search
res = hybrid_search("How do I set up AirSync?", k_final=5)
for r in res:
    print(r["score"], r["id"], r["title"][:50])


0.87890625 ART-2045_KNOWLEDGE_NODE-29 AirSync | Snap-ins | DevRev
0.765625 ART-16807_KNOWLEDGE_NODE-33 BrowserStack AirSync | AirSync | Snap-ins | DevRev
0.69140625 ART-2045_KNOWLEDGE_NODE-60 AirSync | Snap-ins | DevRev
0.6875 ART-4272_KNOWLEDGE_NODE-27 OneDrive AirSync | AirSync | Snap-ins | DevRev
0.61328125 ART-17215_KNOWLEDGE_NODE-6 Local development | DevRev | Docs


## 7. Evaluate on Annotated Queries
Calculate Recall@K

In [ ]:
import numpy as np
from tqdm import tqdm

def evaluate_true(annotated_df, search_fn, Ks=[1, 3, 5, 10]):
    results = {f"recall@{k}": [] for k in Ks}
    results.update({f"mrr@{k}": [] for k in Ks})
    ndcg_scores = []

    for _, row in tqdm(annotated_df.iterrows(), total=len(annotated_df), desc="Evaluating"):
        golden_ids = {r["id"] for r in row["retrievals"]}
        max_k = max(Ks)
        
        res = search_fn(row["query"], k_final=max_k)
        preds = [r["id"] for r in res]

        for k in Ks:
            preds_k = preds[:k]
            # Recall@K
            results[f"recall@{k}"].append(
                1 if golden_ids & set(preds_k) else 0
            )
            # MRR@K
            mrr = 0.0
            for rank, pid in enumerate(preds_k, 1):
                if pid in golden_ids:
                    mrr = 1 / rank
                    break
            results[f"mrr@{k}"].append(mrr)

        # nDCG@10
        gains = [1 if pid in golden_ids else 0 for pid in preds[:10]]
        dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
        ideal = sum(1 / np.log2(i + 2) for i in range(min(len(golden_ids), 10)))
        ndcg_scores.append(dcg / ideal if ideal > 0 else 0)

    print("=" * 40)
    for k in Ks:
        print(f"Recall@{k:<3} {np.mean(results[f'recall@{k}']):.4f}")
        print(f"MRR@{k:<5} {np.mean(results[f'mrr@{k}']):.4f}")
    print(f"nDCG@10   {np.mean(ndcg_scores):.4f}")
    print("=" * 40)
    return results

# Run the evaluation on the annotated set
evaluate_true(annotated_df, hybrid_search, Ks=[1, 3, 5, 10])


## 8. Generate Output for Test Queries

In [26]:
test_results = []
TOP_K = 10
    
for item in tqdm(test_queries["test"], desc="Processing test queries"):
    query_id = item["query_id"]
    query = item["query"]
    
    res = hybrid_search(query, k_final=TOP_K)
    retrievals = [{"id": r["id"], "title": r["title"], "text": r["text"]} for r in res]
    
    test_results.append({
        "query_id": query_id,
        "query": query,
        "retrievals": retrievals
    })

with open("test_queries_results.json", "w") as f:
    json.dump(test_results, f, indent=2)

print("Results saved to test_queries_results.json")


Processing test queries: 100%|██████████| 92/92 [01:21<00:00,  1.12it/s]

Results saved to test_queries_results.json


## 9. Comprehensive Evaluation on Annotated Queries (nDCG & MRR)

In [ ]:
import json
import numpy as np
from tqdm import tqdm

annotated = [row for row in annotated_queries["train"]]

def evaluate_true(annotated_data, search_fn, Ks=[1, 3, 5, 10]):
    results = {f"recall@{k}": [] for k in Ks}
    results.update({f"mrr@{k}": [] for k in Ks})
    ndcg_scores = []

    for item in tqdm(annotated_data, desc="Evaluating"):
        golden_ids = {r["id"] for r in item["retrievals"]}
        max_k = max(Ks)
        preds = [r["id"] for r in search_fn(item["query"], k_final=max_k)]

        for k in Ks:
            preds_k = preds[:k]
            # Recall@K
            results[f"recall@{k}"].append(
                1 if set(golden_ids) & set(preds_k) else 0
            )
            # MRR@K
            mrr = 0.0
            for rank, pid in enumerate(preds_k, 1):
                if pid in golden_ids:
                    mrr = 1 / rank
                    break
            results[f"mrr@{k}"].append(mrr)

        # nDCG@10
        gains = [1 if pid in golden_ids else 0 for pid in preds[:10]]
        dcg = sum(g / np.log2(i + 2) for i, g in enumerate(gains))
        ideal = sum(1 / np.log2(i + 2) for i in range(min(len(golden_ids), 10)))
        ndcg_scores.append(dcg / ideal if ideal > 0 else 0)

    print("=" * 40)
    for k in Ks:
        print(f"Recall@{k:<3} {np.mean(results[f'recall@{k}']):.4f}")
        print(f"MRR@{k:<5} {np.mean(results[f'mrr@{k}']):.4f}")
    print(f"nDCG@10   {np.mean(ndcg_scores):.4f}")
    print("=" * 40)
    return results

metrics = evaluate_true(annotated, hybrid_search, Ks=[1, 3, 5, 10])

Evaluating: 100%|██████████| 291/291 [04:44<00:00,  1.02it/s]

Recall@1   0.5155
MRR@1     0.5155
Recall@3   0.6323
MRR@3     0.5693
Recall@5   0.6632
MRR@5     0.5760
Recall@10  0.7010
MRR@10    0.5815
nDCG@10   0.3861
